# Extra X2 · Several agents in one graph, designed for cost and latency

**Time:** 55 min · **Work in teams of 3** · **Optional lab:** use it when there is spare time after Lab 21.

**Where we are in the day:** in Day 2 (Labs 10 to 15) you built LangGraph graphs with the small model inside the nodes. On Day 3 you made the model small and fast, and in Lab 20 you used the model's own confidence to decide when to ask a bigger model.
**Today's question:** when we split one job between several agents, what does the design cost in model calls, tokens and waiting time, and when is a second agent worth it?
**What you will do:**
- Build the **same task five ways** on the home assistant: one agent with all tools; a supervisor with three specialists; a router made of plain code; a fan-out of parallel sub-tasks; and a cascade from a small to a bigger model.
- Run each design on the same **20 requests** and write down four numbers: model calls per request, tokens, time and accuracy. We also add one **cost unit** (an assumption, explained below).
- Read the table, write four design rules from it, and change one design parameter and measure again.

**What you will see:** one table with five rows (and a sixth variant). The design with the most agents is not the best by default, and the cheapest design is not always the most accurate. The table shows the trade, and you decide.
**Why it matters on a device:** every model call costs time, memory and battery on a small device. The number and the size of the calls decide whether the user waits for half a second or for four.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- All the cells together take about **4 minutes** on an RTX 4090, once the models are on the disk. The designs with many model calls take the longest.
- This lab needs a GPU with about 10 GB free: it loads the 1.5-billion-parameter model (the small one) and the 3-billion-parameter model of Day 2 (the big one). If a model is missing, it downloads first, which takes longer.
- If a cell runs much longer than the time we say, press the stop button (interrupt the kernel) and ask the trainer. Do not start it again and again.
- Before you start: Labs 10, 11 and 20 should be done. This lab does **not** need the trained model of Lab 17: it uses the plain 1.5B model, so its numbers are not the numbers of Labs 16 to 20.

**Words used in this lab:**
- **Agent:** here, a model call (or a short loop of model calls) with its own instructions and its own tools.
- **Supervisor:** an agent whose only job is to choose which agent works next.
- **Router:** the part of the graph that chooses the path. A router can be a model, or plain code.
- **Fan-out and fan-in:** split a job into sub-tasks that do not depend on each other, run them at the same time (fan-out), then collect the answers (fan-in).
- **Cascade:** the small model answers first; a bigger model is asked only when the small one is not sure.
- **Latency:** how long the user waits for the answer. Here: milliseconds per request.
- **Cost unit:** our own measure of the price of a design. It is explained in X2.2 and it is an **assumption for teaching**, never a real price.

**The idea of the lab:** *a model call is the expensive step, so a good design asks for the fewest calls, with the shortest prompts, to the smallest model that gets the job right.*

**Python in this lab:** loops, `if`, lists, dictionaries and the LangGraph pieces of Day 2 (state, nodes, edges, `Command`, `Send`). The measuring code is in `x_agents.py`.

In [ ]:
import sys, os, json
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import operator
import pandas as pd
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
import home_domain as hd
import x_agents as xa
from graph_viz import draw_graph
from slm_lab import check, OUT

Now load the two models and warm them up with one short call each, so that the first design does not pay for the start-up.

In [ ]:
xa.load_models()
xa.set_current("warm-up", 0)
for size in ["small", "big"]:
    xa.MeteredModel(size, max_new_tokens=5).invoke([HumanMessage("Reply with the word ready.")])
xa.clear_log("warm-up")
print("small model:", xa.SMALL_ID)
print("big model  :", xa.BIG_ID)
print("GPU memory held by the models:", xa.gpu_memory_gb(), "GB")

**You should see** the two model names and the memory they hold together (a few GB for the small one, about twice as much for the big one).

*In one sentence: the small model is the one we would put on the device, and the big one is the cloud model that we want to call as rarely as possible.*

## X2.1 The task: the same 20 requests for every design
The task is the smart home of Day 2: lights, blinds, air conditioners, scenes and the front door, with the same tools and the same checks (`home_domain.py`). The model **proposes** a tool call, the code **validates** it, and a risky call (unlock the door, the `away` scene) is **refused until a person says yes**.

We fix **20 requests**. They are not all easy: some have two or three commands in one sentence, some are Hinglish or full of speech-recognition mistakes, some are not a job of this assistant (a joke, a TV channel), and two are risky.

In [ ]:
rows = []
for index in range(len(xa.REQUESTS)):
    request = xa.REQUESTS[index]
    rows.append({"no": index + 1, "kind": request["kind"], "request": request["text"], "right answer": xa.expected_text(index)})
pd.DataFrame(rows)

**You should see** the 20 requests with the right answer for each. Request 19 and 20 are risky: the right answer is that **nothing changes**, because no person said yes.

**How we score accuracy, by code and not by eye.** Each request starts from a home with the devices it needs already set (for example the balcony light is on before "switch off the balcony light"). A request is **right** only when the home ends in **exactly** the expected state. A design that does nothing is right only for the requests whose right answer is "nothing changes". One request is 5 per cent of the score, so treat differences of 5 or 10 points as a hint, not as proof.

In [ ]:
example = 6
print("request:", xa.REQUESTS[example]["text"])
print("the home must end with these changes:", xa.changes(xa.expected_home(example)))
check(len(xa.REQUESTS) == 20, "Checkpoint: there are 20 requests, each with a right final state.",
      "The request list is not 20 long. Do not edit x_agents.py; restart the kernel and run the cells again.")

## X2.2 What we measure, and the cost unit
For every design we measure, on the same 20 requests:

| Number | Meaning |
|---|---|
| **calls per request** | how many times the model was called (a cache hit is not a call) |
| **tokens per request** | prompt tokens plus answer tokens of all the calls |
| **cost units per request** | our price of those tokens (below) |
| **ms per request** | wall-clock time from the request to the final state of the home |
| **accuracy %** | requests whose final home state is exactly right |

**The cost unit is an ASSUMPTION for teaching.** We do not know the price of running a model on your product, so we use a simple rule that keeps the right order: a bigger model costs more per token, and an answer token costs more than a prompt token (the model writes it one token at a time).

*cost units of a call = (prompt tokens + 3 x answer tokens) x size weight / 1000*, with size weight **1.0** for the small model and **2.0** for the big one.

These weights are **invented**. They are not real prices. Every model call is kept in a log, so you can change the weights and calculate the table again **without running a model**.

In [ ]:
print("size weights (ASSUMPTION):", xa.SIZE_WEIGHT)
print("an answer token counts", xa.OUT_WEIGHT, "prompt tokens (ASSUMPTION)")
example_call = {"size": "small", "prompt_tokens": 400, "new_tokens": 30}
print("a small-model call with 400 prompt tokens and 30 answer tokens costs", xa.entry_cost(example_call), "units")
example_call["size"] = "big"
print("the same call on the big model costs", xa.entry_cost(example_call), "units")

**You should see** that the same call costs twice as much on the big model. This is only the rule of the lab; it is not a price.

**Predict.** Before we build anything, write on paper the five designs in the order you expect, from **fewest model calls per request** to **most**. Also write which one you think is the **most accurate**. We compare with the table at the end.

1. one agent with all six tools
2. a supervisor with three specialists, every step a model call
3. a router made of plain code, with specialists only when needed
4. a fan-out of the sub-tasks, in parallel
5. a cascade: small model first, bigger model only when unsure

## X2.3 Design 1: one agent with all the tools
The simplest design is the one of Lab 9: **one agent**, the six tools in its prompt, and a loop. The model calls tools, the tools run, the model reads the results and says what was done. In LangGraph this is two nodes. The `Command` object lets a node **update the state and choose the next node in one step** (Lab 11).

In [ ]:
class AgentState(TypedDict, total=False):
    request: str
    home: dict
    messages: Annotated[list, operator.add]     # the whole conversation: every update is appended
    steps: int
    reply: str

one_agent_model = xa.MeteredModel("small", max_new_tokens=100).bind_tools(hd.TOOLS)
print("tools in the prompt of this agent:", len(hd.TOOLS))

In [ ]:
def agent(state: AgentState) -> Command[Literal["tools", "__end__"]]:
    first = []
    if len(state.get("messages", [])) == 0:
        first = [SystemMessage(xa.ALL_TOOLS_SYSTEM), HumanMessage(state["request"])]
    reply = one_agent_model.invoke(state.get("messages", []) + first)
    update = {"messages": first + [reply], "steps": state.get("steps", 0) + 1}
    if len(reply.tool_calls) > 0 and update["steps"] < 3:
        return Command(update=update, goto="tools")          # the model asked for tools: run them, then come back
    update["reply"] = reply.content
    return Command(update=update, goto=END)

The `tools` node is **code**: it runs each call through `run_tool` (validate, refuse the risky ones, execute), and gives the results back to the agent as tool messages.

In [ ]:
def tools(state: AgentState) -> Command[Literal["agent"]]:
    home = state["home"]
    answers = []
    for call in state["messages"][-1].tool_calls:
        home, message, status = xa.run_tool(home, call["name"], call["args"])
        answers.append(ToolMessage(content=message, tool_call_id=call["id"]))
    return Command(update={"home": home, "messages": answers}, goto="agent")

g1 = StateGraph(AgentState)
g1.add_node("agent", agent)
g1.add_node("tools", tools)
g1.add_edge(START, "agent")
app1 = g1.compile()
draw_graph(app1, "Design 1: one agent with all six tools")

**You should see** two boxes, `agent` and `tools`, with arrows in both directions: a loop. The `Command` decides the way, so the arrows come from the code of the nodes.

**Predict.** Request 7 has two commands: "turn on the living room light and set the living room ac to 24". How many **model calls** will this design make? Write a number. Then run the trace: it prints what each node returned.

In [ ]:
xa.trace(app1, 6)
print()
xa.ledger("trace", 0)

**You should see** the nodes in order (`agent`, `tools`, `agent`) and below them the **ledger**: one line per model call with its size, its tokens, its time and how sure the model was (the smallest probability it gave to any token of its answer; the idea of Lab 20). In a normal run the agent calls the model twice: once to choose the tools, once to say what was done. Look at the prompt tokens of both calls: **the six tool descriptions are sent every time**.

Now the same design on all 20 requests. It takes about 40 seconds.

In [ ]:
results = {}
results["1 one agent"] = xa.run_design("1 one agent", app1)
print(xa.summarise("1 one agent", results["1 one agent"]))

In [ ]:
check(len(results["1 one agent"]) == 20, "Checkpoint: design 1 ran on all 20 requests.",
      "Design 1 did not run on 20 requests. Run the cells of X2.3 again, in order.")

**You should see** one row of numbers. Write down the calls, the tokens and the accuracy of design 1: it is the **baseline** that the other four must beat on something.

## X2.4 Design 2: a supervisor and three specialists
Now we split the work. A **supervisor** reads the request and chooses a specialist: **lights** (lights and blinds), **climate** (air conditioners) or **house** (scenes and the front door). The specialist calls its own tools, then reports back to the supervisor, who chooses the next one or says `finish`. **Every step is a model call**, including the supervisor's.

Two things to notice. Each specialist has **only two tools** in its prompt, so its prompt is shorter than the one of design 1. And the supervisor must be called **once more than the number of specialists** that work.

In [ ]:
class TeamState(TypedDict, total=False):
    request: str
    home: dict
    done: Annotated[list, operator.add]       # the specialists that have finished
    calls: Annotated[list, operator.add]      # the tool calls they proposed
    steps: int
    reply: str

supervisor_model = xa.MeteredModel("small", max_new_tokens=6)
team_models = {}
for domain in xa.DOMAINS:
    team_models[domain] = xa.MeteredModel("small", max_new_tokens=60).bind_tools(xa.tools_for(xa.DOMAINS[domain]["tools"]))

The supervisor writes one word. A rule in **code** stops it from sending the same specialist twice, and from looping for ever (the lesson of Lab 11: the model proposes, the code decides).

In [ ]:
def supervisor(state: TeamState) -> Command[Literal["lights", "climate", "house", "finish"]]:
    done = state.get("done", [])
    messages = [SystemMessage(xa.supervisor_system(done)), HumanMessage(state["request"])]
    choice = xa.pick_word(supervisor_model.invoke(messages).content, ["lights", "climate", "house", "finish"])
    steps = state.get("steps", 0) + 1
    if choice in done or steps > 5:
        choice = "finish"                                  # a rule in code: never the same specialist twice
    return Command(update={"steps": steps}, goto=choice)

def run_specialist(state, domain):
    reply = team_models[domain].invoke([SystemMessage(xa.specialist_system(domain)), HumanMessage(state["request"])])
    return {"calls": xa.calls_of(reply), "done": [domain]}

The three specialists are the same function with a different name. The `finish` node is code only: it **runs** all the proposed calls and writes the reply from the results. We will reuse it in the next designs.

In [ ]:
def lights(state: TeamState) -> dict:
    return run_specialist(state, "lights")

def climate(state: TeamState) -> dict:
    return run_specialist(state, "climate")

def house(state: TeamState) -> dict:
    return run_specialist(state, "house")

def finish(state) -> dict:
    home, messages = xa.apply_calls(state["home"], state.get("calls", []))
    return {"home": home, "reply": xa.make_reply(messages)}

In [ ]:
g2 = StateGraph(TeamState)
for name, function in [("supervisor", supervisor), ("lights", lights), ("climate", climate), ("house", house), ("finish", finish)]:
    g2.add_node(name, function)
g2.add_edge(START, "supervisor")
for name in ["lights", "climate", "house"]:
    g2.add_edge(name, "supervisor")                       # every specialist reports back to the supervisor
g2.add_edge("finish", END)
app2 = g2.compile()
draw_graph(app2, "Design 2: supervisor and three specialists")

**You should see** the supervisor in the middle, three specialists that return to it, and the `finish` box.

**Predict.** How many model calls will request 10 need ("set the study ac to 21, close the study blinds and turn on the study light")? It has three commands in two domains. Write a number, then run the trace. This design is the **chattiest**: the run on all 20 requests, further down, takes about a minute.

In [ ]:
xa.trace(app2, 9)
print()
xa.ledger("trace", 0)

**You should see** the supervisor called again and again, with a specialist between each pair of calls. Count the lines of the ledger.

**How big is a specialist's prompt?** We can measure it without running the model: the same request with the six tools of design 1, and with the two tools of the lights specialist.

In [ ]:
chat = [{"role": "system", "content": xa.ALL_TOOLS_SYSTEM}, {"role": "user", "content": xa.REQUESTS[0]["text"]}]
print("prompt with all six tools :", xa.prompt_size("small", chat, hd.TOOLS), "tokens")
print("prompt with the two tools of one specialist:", xa.prompt_size("small", chat, xa.tools_for(["set_light", "set_blinds"])), "tokens")

**You should see** that the specialist's prompt is clearly shorter. A shorter prompt is cheaper, and a small model that sees fewer tools also has fewer wrong tools to choose. This is the **real benefit of specialists**: not the extra agents, but the smaller prompt each one reads. Whether it pays for the extra calls is what the table will tell.

Now run all 20 requests.

In [ ]:
results["2 supervisor"] = xa.run_design("2 supervisor", app2)
print(xa.summarise("2 supervisor", results["2 supervisor"]))

In [ ]:
check(len(results["2 supervisor"]) == 20, "Checkpoint: design 2 ran on all 20 requests.",
      "Design 2 did not run on 20 requests. Run the cells of X2.4 again, in order.")

## X2.5 Design 3: a router made of plain code
A model is not needed to read "turn on the kitchen light". A few **rules** can do it: cut the request at "and" and commas, look for a device word (light, blinds, ac, scene, door), a room, "on" or "off", a number. If the rules find the device **and every value it needs**, the tool call is made by code, with **zero model calls**. Only the commands the rules cannot read are sent to a **specialist** (when the rules at least found the device) or to the general agent (when they found nothing).

Before we run any model, count how many of the commands in our 20 requests the rules can read completely. This cell uses no model at all.

In [ ]:
total = 0
by_rules = 0
for request in xa.REQUESTS:
    for part in xa.split_request(request["text"]):
        total = total + 1
        if xa.code_parse(part)["complete"]:
            by_rules = by_rules + 1
print("commands in the 20 requests:", total)
print("commands the rules can read completely:", by_rules)
print("example, one command:", xa.code_parse("bedroom ka ac 23 par kar do"))

**You should see** that the rules read most of the commands, even a Hinglish one ("ka ac 23 par"). The ones left are the speech-recognition mistake ("lite"), a Hindi word the rules do not know ("parde"), and the two requests that are not a job of this assistant. Notice that the rules also read the **risky** commands; the code then refuses to run them. Rules and checks in code are free and always give the same answer.

In [ ]:
class RuleState(TypedDict, total=False):
    request: str
    home: dict
    leftovers: list                           # the commands the rules could not read completely
    calls: Annotated[list, operator.add]
    reply: str

The `route` node is plain code. For every command it asks the rules; a command that the rules read completely becomes a tool call at once.

In [ ]:
def route(state: RuleState) -> dict:
    calls = []
    leftovers = []
    for part in xa.split_request(state["request"]):
        found = xa.code_parse(part)
        if found["complete"]:
            calls = calls + found["calls"]               # the rules read it: no model call
        else:
            leftovers.append({"part": part, "domain": found["domain"]})
    return {"calls": calls, "leftovers": leftovers}

After `route`, a **conditional edge** chooses the path: model help for the leftovers, or straight to `finish`. The specialist node calls the model once per leftover command, with the specialist's two tools if the device is known, and with all six tools if it is not.

In [ ]:
def after_route(state: RuleState) -> str:
    if len(state["leftovers"]) > 0:
        return "specialist"
    return "finish"

helper_models = {"": one_agent_model}
for domain in xa.DOMAINS:
    helper_models[domain] = team_models[domain]

In [ ]:
def specialist(state: RuleState) -> dict:
    calls = []
    for item in state["leftovers"]:
        system = xa.ALL_TOOLS_SYSTEM
        if item["domain"] != "":
            system = xa.specialist_system(item["domain"])
        reply = helper_models[item["domain"]].invoke([SystemMessage(system), HumanMessage(item["part"])])
        calls = calls + xa.calls_of(reply)
    return {"calls": calls}

In [ ]:
g3 = StateGraph(RuleState)
for name, function in [("route", route), ("specialist", specialist), ("finish", finish)]:
    g3.add_node(name, function)
g3.add_edge(START, "route")
g3.add_conditional_edges("route", after_route, {"specialist": "specialist", "finish": "finish"})
g3.add_edge("specialist", "finish")
g3.add_edge("finish", END)
app3 = g3.compile()
draw_graph(app3, "Design 3: rules first, model only for what the rules cannot read")

**You should see** a short path (`route`, `finish`) and a longer one through `specialist`. Most requests take the short one.

**Predict.** For how many of the 20 requests will this design make **no model call at all**? Use your count from above. Then run it; this one is quick.

In [ ]:
results["3 rules first"] = xa.run_design("3 rules first", app3)
print(xa.summarise("3 rules first", results["3 rules first"]))
silent = 0
for index in range(20):
    if len(xa.entries_for("3 rules first", index)) == 0:
        silent = silent + 1
print("requests with no model call at all:", silent, "of 20")

In [ ]:
check(len(results["3 rules first"]) == 20 and silent > 0, "Checkpoint: design 3 ran on all 20 requests.",
      "Design 3 did not run on 20 requests. Run the cells of X2.5 again, in order.")

## X2.6 Design 4: fan-out of the independent sub-tasks
"Turn on the kitchen light **and** set the study ac to 22" has two commands that do not depend on each other. A **planner** (one model call) writes one line per command and names the domain. Then LangGraph starts **one worker per line, at the same time**, using `Send` (Day 2, Lab 11). The workers only **propose** tool calls; one `finish` node runs them in code, so two workers never change the home at the same moment.

The calls of the workers are collected with a **reducer** (`operator.add`), as in Lab 11.

In [ ]:
class FanState(TypedDict, total=False):
    request: str
    home: dict
    plan: list                                # (domain, command) pairs written by the planner
    calls: Annotated[list, operator.add]      # every worker adds its calls
    reply: str

class WorkerInput(TypedDict):
    domain: str
    part: str

The planner is one model call. Its answer is read by `parse_plan`, which keeps only well-formed lines.

In [ ]:
planner_model = xa.MeteredModel("small", max_new_tokens=80)

def plan(state: FanState) -> dict:
    reply = planner_model.invoke([SystemMessage(xa.PLAN_SYSTEM), HumanMessage(state["request"])])
    return {"plan": xa.parse_plan(reply.content)}

In [ ]:
def fan_out(state: FanState):
    sends = []
    for domain, part in state["plan"]:
        if domain in xa.DOMAINS:
            sends.append(Send("worker", {"domain": domain, "part": part}))      # one worker per sub-task
    if len(sends) == 0:
        return "finish"                                  # nothing for the home to do
    return sends

def worker(state: WorkerInput) -> dict:
    reply = team_models[state["domain"]].invoke([SystemMessage(xa.specialist_system(state["domain"])), HumanMessage(state["part"])])
    return {"calls": xa.calls_of(reply)}

In [ ]:
g4 = StateGraph(FanState)
for name, function in [("plan", plan), ("worker", worker), ("finish", finish)]:
    g4.add_node(name, function)
g4.add_edge(START, "plan")
g4.add_conditional_edges("plan", fan_out, ["worker", "finish"])
g4.add_edge("worker", "finish")                          # fan-in: finish waits for all the workers
g4.add_edge("finish", END)
app4 = g4.compile()
draw_graph(app4, "Design 4: plan, then one worker per sub-task in parallel")

**Predict.** Two questions. (a) How many model calls will request 10 (three commands) need now? (b) The workers run "at the same time". Will the request take **less time** than in design 2? And will it take **a third** of the time of running the three workers one after the other? Write your answers.

In [ ]:
xa.trace(app4, 9)
print()
xa.ledger("trace", 0)

**You should see** the planner's lines, then the workers, then `finish`, and in the ledger one planner call and one call per worker. The workers share **one GPU**, so parallel calls do not give three times the speed: the GPU is one chip that does one thing at a time. Parallel calls help most when the calls **wait** (for a network or a slow tool). On one local GPU, the better way to run sub-tasks together is a **batch**: one `generate` call that carries all the prompts. We add that as a variant, **4b**: the same planner, then **one batched call** for all the workers.

In [ ]:
def batch(state: FanState) -> dict:
    return {"calls": xa.batch_workers(state["plan"])}      # all the workers in ONE generate call

g4b = StateGraph(FanState)
for name, function in [("plan", plan), ("batch", batch), ("finish", finish)]:
    g4b.add_node(name, function)
g4b.add_edge(START, "plan")
g4b.add_edge("plan", "batch")
g4b.add_edge("batch", "finish")
g4b.add_edge("finish", END)
app4b = g4b.compile()

Run both. Together they take about a minute and a half.

In [ ]:
results["4 fan-out (parallel)"] = xa.run_design("4 fan-out (parallel)", app4)
results["4b fan-out (one batch)"] = xa.run_design("4b fan-out (one batch)", app4b)
for name in ["4 fan-out (parallel)", "4b fan-out (one batch)"]:
    print(xa.summarise(name, results[name]))

In [ ]:
check(len(results["4 fan-out (parallel)"]) == 20 and len(results["4b fan-out (one batch)"]) == 20,
      "Checkpoint: design 4 and design 4b ran on all 20 requests.",
      "Design 4 did not run on 20 requests. Run the cells of X2.6 again, in order.")

**You should see** two rows with the **same tokens**: the batch does the same work. Compare the **calls** (the batch counts as one call, with several prompts inside) and the **ms per request**. A call is not a price: the **tokens** are.

## X2.7 Design 5: a cascade from the small to the big model
In Lab 20 the small model's **confidence** told us when to ask a bigger model. Here we do the same with the tool call. The small model answers first, with all six tools, in **one call**, and **code** writes the reply from the tool results (no summary call). Then `decide` looks at two things, **in this order**:

1. **Code check:** are all the proposed calls allowed (known tool, known room, a temperature from 16 to 30)? If not, ask the big model.
2. **Confidence:** the smallest probability the small model gave to any token of its answer. If it is **below the threshold**, ask the big model.

Otherwise we keep the small model's answer. The big model is the exception, not the rule.

In [ ]:
class CascadeState(TypedDict, total=False):
    request: str
    home: dict
    calls: list                               # the calls we will run (the big model's replace the small one's)
    small_calls: list                         # what the small model proposed (kept to measure it)
    text: str
    confidence: float
    escalated: bool
    reply: str

settings = {"threshold": 0.9}                 # Change this: below this confidence we ask the big model
big_agent_model = xa.MeteredModel("big", max_new_tokens=100).bind_tools(hd.TOOLS)

In [ ]:
def small_try(state: CascadeState) -> dict:
    reply = one_agent_model.invoke([SystemMessage(xa.ALL_TOOLS_SYSTEM), HumanMessage(state["request"])])
    calls = xa.calls_of(reply)
    return {"calls": calls, "small_calls": calls, "text": reply.response_metadata["raw"], "confidence": reply.response_metadata["confidence"]}

def decide(state: CascadeState) -> str:
    if not xa.calls_valid(state["calls"], state["text"]):
        return "big_try"                                  # the code does not accept the small model's answer
    if state["confidence"] < settings["threshold"]:
        return "big_try"                                  # the small model is not sure enough
    return "finish"

def big_try(state: CascadeState) -> dict:
    reply = big_agent_model.invoke([SystemMessage(xa.ALL_TOOLS_SYSTEM), HumanMessage(state["request"])])
    return {"calls": xa.calls_of(reply), "escalated": True}

In [ ]:
g5 = StateGraph(CascadeState)
for name, function in [("small_try", small_try), ("big_try", big_try), ("finish", finish)]:
    g5.add_node(name, function)
g5.add_edge(START, "small_try")
g5.add_conditional_edges("small_try", decide, {"big_try": "big_try", "finish": "finish"})
g5.add_edge("big_try", "finish")
g5.add_edge("finish", END)
app5 = g5.compile()
draw_graph(app5, "Design 5: the small model first, the big model only when unsure")

**Predict.** Will the cascade be **more accurate** than design 1 (which uses only the small model), **equal**, or **less**? And will it cost **more** or **less** than design 1? Write both guesses. One thing to notice before you answer: design 5 also drops the final "say what was done" call that design 1 makes.

In [ ]:
results["5 cascade"] = xa.run_design("5 cascade", app5)
print(xa.summarise("5 cascade", results["5 cascade"]))

In [ ]:
check(len(results["5 cascade"]) == 20, "Checkpoint: design 5 ran on all 20 requests.",
      "Design 5 did not run on 20 requests. Run the cells of X2.7 again, in order.")

### How sure was the small model?
This is the table of Lab 20, built from the run we just made. For each request we look at the small model's **first** answer: was the home state right if we had run it, and how sure was the model?

In [ ]:
small_rows = []
for result in results["5 cascade"]:
    if result["small_calls"] is not None and result["confidence"] is not None:
        small_right = xa.state_after(result["index"], result["small_calls"]) == xa.expected_home(result["index"])
        small_rows.append({"index": result["index"], "confidence": result["confidence"], "small_ok": small_right})
print("requests with a small-model answer:", len(small_rows))

In [ ]:
sum_right = 0.0
count_right = 0
sum_wrong = 0.0
count_wrong = 0
for row in small_rows:
    if row["small_ok"]:
        sum_right = sum_right + row["confidence"]
        count_right = count_right + 1
    else:
        sum_wrong = sum_wrong + row["confidence"]
        count_wrong = count_wrong + 1
print("small model right:", count_right, "requests, average confidence", round(sum_right / max(count_right, 1), 3))
print("small model wrong:", count_wrong, "requests, average confidence", round(sum_wrong / max(count_wrong, 1), 3))

**You should see** two averages. The question of Lab 20 again: is the model **less sure when it is wrong**? If the gap is small, confidence is a weak signal for this task, and the code check does more of the work. Now the table for several thresholds: how many requests would be sent to the big model, and how many wrong small answers would we still keep?

In [ ]:
threshold_rows = []
for threshold in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    sent_up = 0
    wrong_kept = 0
    for row in small_rows:
        if row["confidence"] < threshold:
            sent_up = sent_up + 1
        elif not row["small_ok"]:
            wrong_kept = wrong_kept + 1
    threshold_rows.append({"threshold": threshold, "sent to the big model": sent_up, "wrong small answers kept": wrong_kept})
pd.DataFrame(threshold_rows)

**You should see** that a higher threshold sends more requests to the big model and keeps fewer wrong small answers. This table counts only the confidence; the code check of `decide` sends some more. There is no free lunch: every request that goes up costs big-model tokens.

## X2.8 The table: five designs, one task
Now the scoreboard. Each row comes from the call log of that design.

In [ ]:
board = []
for name in results:
    board.append(xa.summarise(name, results[name]))
pd.DataFrame(board)

In [ ]:
check(len(board) == 6, "Checkpoint: the table has 6 rows (five designs and the batch variant).",
      "The table has fewer than 6 rows. Run the design cells above again, in order.")
by_name = {}
for row in board:
    by_name[row["design"]] = row
check(by_name["3 rules first"]["calls per request"] < by_name["2 supervisor"]["calls per request"],
      "Design 3 makes fewer model calls than design 2: code did the routing for free.",
      "Design 3 should call the model less than design 2. Look at the leftovers of design 3 with xa.show_mistakes.")

**Read the table like an engineer, not like a student.** One request is 5 per cent, and the cost unit is our assumption, so look for **big** differences.

- Which design makes the **fewest model calls**? Which makes the **most**?
- Which design reads the **fewest tokens**? Is it the same one? (Remember that every call sends the tool descriptions again.)
- Which design is the **fastest** in ms per request, and which is the **slowest**?
- Which design is the **most accurate**? Is it also cheap?
- Is there a design that is **both** cheaper than design 1 **and** at least as accurate? That is a design worth keeping.

Now the accuracy by kind of request, and where each design went wrong.

In [ ]:
kind_rows = []
for name in results:
    scores = xa.accuracy_by_kind(results[name])
    scores["design"] = name
    kind_rows.append(scores)
pd.DataFrame(kind_rows)

In [ ]:
for name in results:
    print("---", name)
    xa.show_mistakes(results[name], limit=3)

**You should see** that the designs fail on **different** requests. Look at the `messy` and `not for this assistant` kinds: they are where rules, planners and a small model can each go wrong in their own way. Ask one question for each mistake: did the **model** read the request wrongly, did the **code** have no rule for it, or did the **design** send it to the wrong place?

Save the table, so that the trainer can compare the runs.

In [ ]:
with open(str(OUT / "x2_multi_agent_table.json"), "w", encoding="utf-8") as f:
    json.dump(board, f, indent=1)
print("saved", len(board), "rows:", OUT / "x2_multi_agent_table.json")

## X2.9 Your four design rules
Look at the table once more and write **four rules** for a small device, each one starting with the words below, and each one supported by a **number from your table**. Write them on paper first.

1. **A second agent is worth it when ...**
2. **Code should do the routing when ...**
3. **Run sub-tasks in parallel when ...**
4. **Cache the model's answer when ...**

*✍️ Your answer:* 

A word on **what to cache**. There are three different things, and only the first can be tried with this lab's code.

- **The answer for an exactly repeated prompt.** Cheap and safe for a reading step, as below.
- **The fixed start of the prompt.** Every call of design 1 sends the same system message and the same tool descriptions. A serving engine can keep the work for that fixed start (the prefix cache of Lab 19) and only read the new words.
- **Never the action.** "Turn on the kitchen light" must run every time, even when the model's reading of it comes from the cache.

The cache of this lab keys on the **exact prompt**. We switch it on, run design 3 twice, and compare. Design 3 makes few model calls, so the numbers are small; the idea is the same for the others.

In [ ]:
xa.set_cache(True)
results["3 rules first, cache pass 1"] = xa.run_design("3 rules first, cache pass 1", app3)
results["3 rules first, cache pass 2"] = xa.run_design("3 rules first, cache pass 2", app3)
xa.set_cache(False)
for name in ["3 rules first, cache pass 1", "3 rules first, cache pass 2"]:
    row = xa.summarise(name, results[name])
    print(name, "| model calls per request:", row["calls per request"], "| cache hits:", row["cache hits"], "| accuracy %:", row["accuracy %"])

**You should see** the second pass with **no model calls and only cache hits**, and the same accuracy. The home changed in both passes, because the **tools ran again**; only the model's reading came from the cache.

In [ ]:
check(xa.summarise("3 rules first, cache pass 2", results["3 rules first, cache pass 2"])["calls per request"] == 0,
      "Checkpoint: the second pass made no model call.",
      "The second pass should have no model call. Run the cache cell again: the cache must be switched on before both passes.")

## X2.10 Your turn: change one design parameter and measure again

### TODO 1: move the threshold of the cascade
The threshold of design 5 decides when the big model is asked. In the table you saw how many wrong small answers each threshold keeps. Choose a **different** threshold from 0 to 1, run design 5 again, and compare accuracy, big-model calls and cost with the first run. Write down what you gave up and what you gained. If you are stuck, open the solution cell below the starter: it picks the threshold from your own table of confidences.

In [ ]:
settings["threshold"] = 0.9                  # TODO: choose another value from 0 to 1 (higher: the big model is asked more often)
results["5 cascade, new threshold"] = xa.run_design("5 cascade, new threshold", app5)
first = xa.summarise("5 cascade", results["5 cascade"])
second = xa.summarise("5 cascade, new threshold", results["5 cascade, new threshold"])
for row in [first, second]:
    print(row["design"], "| accuracy %:", row["accuracy %"], "| big-model calls:", row["big-model calls"], "| cost units:", row["cost units per request"])

In [ ]:
check(len(results["5 cascade, new threshold"]) == 20, "Checkpoint: design 5 ran again with your threshold.",
      "Design 5 did not run on 20 requests. Run the starter or the solution cell again.")

**You should see** two rows. A higher threshold usually moves accuracy up and cost up together; a lower one does the opposite. If the accuracy did not change but the cost went down, the threshold was higher than it needed to be. If almost every request went to the big model, the confidence could not separate the right answers from the wrong ones: then no threshold can fix it, and the code check or a better prompt must do the work.

### TODO 2: teach the rules two Hindi words
Request 16 ("bedroom ke parde khol do aur study ki light jala do") is only half read by the rules: they do not know **parde** (blinds) or **khol** (open). Add the missing words to the lists of the rules router, check that the rules now read the command completely, and run design 3 again. The number of model calls should go **down**. If you are stuck, open the solution cell below.

In [ ]:
xa.BLIND_WORDS.append("")                    # TODO: the Hindi word for blinds in request 16
xa.OPEN_WORDS.append("")                     # TODO: the Hindi word for "open" in request 16
parse = xa.code_parse("bedroom ke parde khol do")
print("the rules read it completely:", parse["complete"], xa.describe_calls(parse["calls"]))
results["3 rules, more words"] = xa.run_design("3 rules, more words", app3)
print(xa.summarise("3 rules, more words", results["3 rules, more words"]))

In [ ]:
check("3 rules, more words" in results, "Checkpoint: design 3 ran again with your new words.",
      "Design 3 did not run again. Run the starter or the solution cell again.")

**You should see** that the command is now read completely and the model is called less. Every word you add to the rules is a model call you do not pay for, but every rule can also be wrong in a way the model would not be. Test it, as you just did.

## What you take away
- The number and the size of the model calls decide the cost. A design with more agents is not better by itself: it must make the prompts shorter, or the answers better, by more than the calls it adds.
- Let plain code route and check whatever a rule can handle, and keep the model for the language the rules cannot read. The code always has the last word on what the home does.
- Sub-tasks that do not depend on each other can run together, but on one local GPU a batch helps more than threads. A cascade needs a signal for "not sure", and the code check is the most reliable one. Cache the reading, never the action.
- Measure every design on the same requests, with the same numbers, and write the table down. Never trust one column.

## Check your understanding
1. A design makes more model calls than design 1 but costs fewer units. How can that be?
*✍️ Your answer:* 
2. Why do the workers of design 4 only propose tool calls, and why does one `finish` node run them?
*✍️ Your answer:* 
3. In the cascade, why does `decide` check the proposed calls with code before it looks at the confidence?
*✍️ Your answer:* 
4. Why may we cache the model's reading of "turn on the kitchen light" but never the action?
*✍️ Your answer:* 

## Next
**Extra X3:** the rules router of design 3 and the model that only reads the language are the two ideas of a full dialogue manager. In X3 we build one for a smart TV, with slots, corrections, confirmations, undo and a PIN, and test it on named scenarios.